Практическое задание 

Сделай:

✅ классификацию Cats vs Dogs через ResNet18.

Шаги:

загрузить pretrained model

заморозить backbone

заменить последний слой:

model.fc = nn.Linear(512, 2)

обучить только classifier

Будем использовать:
====

PyTorch

Jupyter Notebook

Cats vs Dogs

ResNet

🧠 Общая идея
=====

- Мы берём pretrained ResNet,
- замораживаем feature extractor,
- дообучаем только classifier.

✅ МОДУЛЬ 0 — Импорт библиотек
===

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms, models
from torch.utils.data import random_split, DataLoader

import matplotlib.pyplot as plt

✅ МОДУЛЬ 1 — Загрузить pretrained model
=====

In [2]:
# Загрузка pretrained ResNet18
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Посмотреть архитектуру
print(model)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

weights=models.ResNet18_Weights.DEFAULT

означает:

👉 загрузить веса, обученные на ImageNet.

То есть модель уже умеет распознавать базовые признаки:
- края
- текстуры
- формы

✅ МОДУЛЬ 2 — Заморозить backbone
=====

In [3]:
for param in model.parameters():
    param.requires_grad = False

Что это делает

Все слои backbone:

❌ не обучаются
❌ веса не меняются

То есть сохраняются pretrained признаки.

✅ МОДУЛЬ 3 — Заменить последний слой
======

In [4]:
#смотрим полседний слой перед изменением
print(model.fc)

Linear(in_features=512, out_features=1000, bias=True)


In [5]:
num_features = model.fc.in_features

model.fc = nn.Linear(num_features, 2)

Почему именно так
===

У original ResNet:

1000 классов (ImageNet)

У нас:

2 класса:
cats / dogs

Проверка
====

In [6]:
print(model.fc)

Linear(in_features=512, out_features=2, bias=True)


✅ МОДУЛЬ 4 — Обучить только classifier
=====

4.0 Загрузка датасета
===

In [7]:
#!pip install kaggle

In [18]:
!kaggle datasets download karakaggle/kaggle-cat-vs-dog-dataset

Dataset URL: https://www.kaggle.com/datasets/karakaggle/kaggle-cat-vs-dog-dataset
License(s): unknown




  0%|          | 0.00/787M [00:00<?, ?B/s]
 17%|#7        | 134M/787M [00:00<00:00, 1.40GB/s]
 35%|###5      | 277M/787M [00:00<00:00, 1.45GB/s]
 53%|#####2    | 416M/787M [00:00<00:00, 838MB/s] 
 70%|#######   | 554M/787M [00:00<00:00, 1.00GB/s]
 85%|########4 | 669M/787M [00:00<00:00, 714MB/s] 
100%|##########| 787M/787M [00:00<00:00, 903MB/s]


Нужно разахивировать kaggle-cat-vs-dog-dataset.zip в папаку \data
=====

4.1 Подготовка данных
====

In [7]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

dataset = datasets.ImageFolder(r"data\kagglecatsanddogs_3367a\PetImages", transform=transform)

print(dataset.classes)

['Cat', 'Dog']


In [8]:
#Разделение train / test - например 80/20

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

#Проверка размеров
print('train_dataset: ',len(train_dataset))
print('test_dataset: ',len(test_dataset))

train_dataset:  19967
test_dataset:  4992


In [9]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

4.2 Оптимизатор только для classifier
===

In [10]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

Почему только classifier
- model.fc.parameters()

Потому что backbone frozen.

4.3 Training loop
===

In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print ('device: ', device)
model = model.to(device)

epochs = 3

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {running_loss / len(train_loader):.4f}")

device:  cuda


C:\Users\dell_Artyom\.conda\envs\torch_old\lib\site-packages\PIL\TiffImagePlugin.py:949: UserWarning: Truncated File Read
  warnings.warn(str(msg))


Epoch 1, Loss: 0.1198
Epoch 2, Loss: 0.0807
Epoch 3, Loss: 0.0765


✅ МОДУЛЬ 5 — Тестирование модели
====

5.1 Загрузка test dataset
===

In [12]:
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

5.2 Evaluation
===

In [13]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Accuracy: {accuracy:.2f}%")

Accuracy: 98.10%
